# 01 - Fundamentos de Gradient Boosting Clasificación y Variantes (XGBoost, LightGBM, CatBoost)

## Curso de Machine Learning: Modelos de Clasificación

Los algoritmos de **Gradient Boosting** representan el estado del arte en el aprendizaje automático para datos tabulares estructurados. A diferencia de los métodos de *Bagging* (como Random Forest), donde múltiples árboles independientes se entrenan en paralelo para reducir la varianza, el **Boosting** construye un ensamble **secuencial** donde cada nuevo árbol de decisión se entrena para corregir los errores residuales (pseudo-residuos) cometidos por los modelos anteriores, reduciendo drásticamente el sesgo.

---

### Contenidos
1. De Bagging a Boosting: Aprendizaje Aditivo Secuencial
2. Formulación Matemática en Clasificación: Log-Odds, Función de Pérdida Log-Loss y Pseudo-Residuos
3. Tasa de Aprendizaje ($\eta$ / `learning_rate`) y Regularización Estructural
4. La Tríada Moderna del Boosting: **XGBoost**, **LightGBM** y **CatBoost**
5. Implementación Práctica en Scikit-Learn y Evolución de la Pérdida (Deviance)
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_hastie_10_2
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report, roc_auc_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas y entorno para Gradient Boosting listos.")


## 1. Aprendizaje Aditivo Secuencial y Descenso de Gradiente Funcional

En un modelo de **Gradient Boosting** con $M$ etapas, la predicción aditiva en el paso $m$ se define como la suma del ensamble anterior más un nuevo árbol base (weak learner) $h_m(\mathbf{x})$ escalado por una tasa de aprendizaje $\eta$:

$$F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \eta \cdot h_m(\mathbf{x})$$

Para **Clasificación Binaria** ($y_i \in \{0, 1\}$), el modelo predice internamente el **log-odds** (logit) $F(\mathbf{x}) = \ln\left(\frac{p}{1-p}\right)$, y la probabilidad estimada $\hat{p}_i$ se obtiene aplicando la función sigmoide $\sigma(z)$:

$$\hat{p}_i = \sigma(F_{m-1}(\mathbf{x}_i)) = \frac{1}{1 + e^{-F_{m-1}(\mathbf{x}_i)}}$$

### Función de Pérdida (Binary Cross-Entropy / Log-Loss):
$$\mathcal{L}(y_i, F(\mathbf{x}_i)) = -\left[ y_i \ln(\hat{p}_i) + (1 - y_i)\ln(1 - \hat{p}_i) \right]$$

### Los Pseudo-Residuos (Gradiente Negativo):
Al derivar la función de pérdida respecto a la predicción en log-odds $F_{m-1}(\mathbf{x}_i)$, obtenemos una expresión sumamente intuitiva para el **pseudo-residuo** $r_{im}$ que el árbol $h_m$ debe aprender a predecir:

$$r_{im} = -\left[ \frac{\partial \mathcal{L}(y_i, F(\mathbf{x}_i))}{\partial F(\mathbf{x}_i)} \right]_{F = F_{m-1}} = y_i - \hat{p}_{i, m-1}$$

Es decir, **cada nuevo árbol de regresión $h_m$ se ajusta directamente sobre la diferencia entre la etiqueta real $y_i \in \{0, 1\}$ y la probabilidad predicha acumulada $\hat{p}_{i, m-1}$**.


## 2. La Tríada Moderna: XGBoost, LightGBM y CatBoost

Aunque `GradientBoostingClassifier` de Scikit-Learn es excelente para comprender el algoritmo clásico de Friedman (2001), en entornos de producción y competencias de Ciencia de Datos dominan tres implementaciones optimizadas:

| Algoritmo | Crecimiento del Árbol | Innovación Principal | Manejo de Categóricas / Nulos | Caso de Uso Ideal |
|---|---|---|---|---|
| **XGBoost** (*eXtreme Gradient Boosting*) | *Depth-wise* (por nivel) | Expansión de Taylor de 2º orden (Gradiente + Hessiano), regularización explícita $L_1$ ($\alpha$) y $L_2$ ($\lambda$), poda por ganancia (`gamma`). | Soporta nulos nativamente (dirección por defecto) y `enable_categorical`. | Estándar general en datos tabulares, alta estabilidad y explicabilidad. |
| **LightGBM** (*Light Gradient Boosting Machine*) | *Leaf-wise* (por hoja de mayor pérdida) | **Histogram-based** binning ($O(\text{bins})$), **GOSS** (muestreo por gradiente) y **EFB** (empaquetado de features exclusivos). | Manejo nativo de categóricas por partición óptima de histogramas y soporte de nulos. | Datasets masivos (millones de filas), entrenamiento ultra-veloz y baja memoria (`HistGradientBoostingClassifier` en Sklearn). |
| **CatBoost** (*Categorical Boosting*) | **Árboles Simétricos** (*Oblivious Trees*) | **Ordered Target Statistics** (evita el *target leakage* en variables categóricas) y árboles balanceados con la misma condición por nivel. | El mejor soporte automático de variables categóricas sin preprocesamiento previo. | Datasets con alta cardinalidad categórica e inferencia ultra-rápida en producción. |


In [ ]:
# Demostracion paso a paso: Como evolucionan los pseudo-residuos en Clasificacion Binaria
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Probabilidad base inicial (F_0): log-odds de la proporcion de la clase positiva
p_inicial = np.mean(y_train)
F_0 = np.log(p_inicial / (1 - p_inicial))
residuos_iniciales = y_train - p_inicial

print(f"Proporcion clase positiva (Benigno=1) en Train: {p_inicial:.4f}")
print(f"Prediccion inicial en Log-Odds (F_0):           {F_0:.4f}")
print(f"Primeros 5 pseudo-residuos iniciales (y - p_0): {np.round(residuos_iniciales[:5], 4)}")


In [ ]:
# Ajuste de GradientBoostingClassifier e inspeccion de la reduccion secuencial de Log-Loss
gbc = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8, # Stochastic Gradient Boosting
    random_state=42
)
gbc.fit(X_train, y_train)

# Calculamos el Log-Loss (Deviance) en cada iteracion m = 1..150 usando staged_predict_proba
train_loss = [log_loss(y_train, y_prob) for y_prob in gbc.staged_predict_proba(X_train)]
test_loss = [log_loss(y_test, y_prob) for y_prob in gbc.staged_predict_proba(X_test)]

mejor_iter = np.argmin(test_loss) + 1
print(f"Exactitud en Test (150 arboles): {accuracy_score(y_test, gbc.predict(X_test))*100:.2f}%")
print(f"ROC-AUC en Test:                 {roc_auc_score(y_test, gbc.predict_proba(X_test)[:, 1]):.4f}")
print(f"Iteracion con menor Log-Loss en Test: Arbol #{mejor_iter} (Log-Loss = {min(test_loss):.4f})")

plt.figure(figsize=(10, 4.8))
plt.plot(range(1, 151), train_loss, label='Train Log-Loss (Deviance)', color='#1f77b4', lw=2)
plt.plot(range(1, 151), test_loss, label='Test Log-Loss (Deviance)', color='#d62728', lw=2)
plt.axvline(mejor_iter, color='black', linestyle='--', label=f'Optimo Test (Iter={mejor_iter})')
plt.xlabel('Numero de Arboles Secuenciales (n_estimators)', fontweight='bold')
plt.ylabel('Binary Cross-Entropy (Log-Loss)', fontweight='bold')
plt.title('Evolucion Secuencial del Error en Gradient Boosting Classifier', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 3. El Efecto de la Tasa de Aprendizaje (`learning_rate`) y `subsample`

Dos mecanismos clave previenen el sobreajuste en el aprendizaje secuencial:
1. **Shrinkage (`learning_rate` $\eta \in (0, 1]$)**: Escala la contribución de cada árbol. Un $\eta$ pequeño (ej. `0.01` - `0.05`) requiere más árboles (`n_estimators`), pero logra una superficie de decisión mucho más suave y generalizable.
2. **Stochastic Gradient Boosting (`subsample < 1.0`)**: Propuesto por Friedman (2002), entrena cada nuevo árbol con una fracción aleatoria sin reemplazo de los datos (ej. `0.8`), combinando la reducción de sesgo del Boosting con la reducción de varianza del Bagging.


In [ ]:
# Comparativa de curvas de aprendizaje para distintas tasas (learning_rate)
tasas = [1.0, 0.2, 0.05, 0.01]
colores = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8']

plt.figure(figsize=(10, 5))
for lr, col in zip(tasas, colores):
    clf_lr = GradientBoostingClassifier(n_estimators=150, learning_rate=lr, max_depth=3, random_state=42)
    clf_lr.fit(X_train, y_train)
    loss_curve = [log_loss(y_test, prob) for prob in clf_lr.staged_predict_proba(X_test)]
    plt.plot(range(1, 151), loss_curve, label=f'learning_rate = {lr}', color=col, lw=2)

plt.xlabel('Numero de Arboles (n_estimators)', fontweight='bold')
plt.ylabel('Test Log-Loss', fontweight='bold')
plt.title('Impacto del Learning Rate en la Convergencia y Sobreajuste', fontweight='bold')
plt.ylim(0, 0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 4. Conclusiones

1. **Corrección Secuencial de Errores**: A diferencia de Random Forest donde los árboles son profundos e independientes, Gradient Boosting utiliza árboles cortos (*weak learners*, típicamente `max_depth` entre 3 y 6) que corrigen iterativamente los pseudo-residuos $y_i - \hat{p}_i$.
2. **Cuidado con el Sobreajuste**: A medida que `n_estimators` crece indefinidamente, el error de entrenamiento tiende a cero pero el modelo puede memorizar el ruido; por ello es vital utilizar **Early Stopping** y regularización.
3. **Ecosistema Moderno**: Mientras que `GradientBoostingClassifier` implementa el algoritmo exacto tradicional, `HistGradientBoostingClassifier` (inspirado en **LightGBM**), **XGBoost** y **CatBoost** escalan este paradigma mediante histogramas y derivadas de segundo orden.

En el siguiente cuaderno, **02_XGBoost_LightGBM_CatBoost_y_Benchmark.ipynb**, implementaremos y compararemos directamente las bibliotecas modernas junto con técnicas de *Early Stopping*, manejo de valores nulos e importancia de características.
